In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tvm
import os

SEED       = 17
BATCH_SIZE = 128
DEVICE     = 'cuda'
EPS        = 8/255
ALPHA      = EPS / 10
PGD_STEPS  = 40

MODEL_NAMES = ["resnet34", "vgg11", "resnet18", "vgg16", "densenet"]
MODEL_PATHS = {n: f'../models/{n}.pth' for n in MODEL_NAMES}
ADV_PATHS   = {n: f'../adv_examples/{n}.pt' for n in MODEL_NAMES}

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

In [28]:
def get_resnet34():
    m = tvm.resnet34(weights=None)
    m.fc = nn.Linear(512, 10)
    return m

def get_vgg11():
    m = tvm.vgg11(weights=None)
    m.classifier[6] = nn.Linear(4096, 10)
    return m

def get_resnet18():
    m = tvm.resnet18(weights=None)
    m.fc = nn.Linear(512, 10)
    return m

def get_vgg16():
    m = tvm.vgg16(weights=None)
    m.classifier[6] = nn.Linear(4096, 10)
    return m

def get_densenet():
    m = tvm.densenet121(weights=None)
    m.classifier = nn.Linear(1024, 10)
    return m

def get_model(name):
    return {
        'resnet34': get_resnet34,
        'vgg11': get_vgg11,
        'resnet18': get_resnet18,
        'vgg16': get_vgg16,
        'densenet': get_densenet
    }[name]()

In [29]:
def fgsm(model, images, labels, criterion):
    images = images.clone().detach().requires_grad_(True)
    model.zero_grad() 
    loss = criterion(model(images), labels)
    loss.backward()
    return (images + EPS * images.grad.sign()).clamp(0, 1).detach()

def pgd(model, images, labels, criterion):
    x = images.clone().detach() + torch.zeros_like(images).uniform_(-EPS, EPS)
    x = x.clamp(0, 1)
    for _ in range(PGD_STEPS):
        x = x.detach().requires_grad_(True)
        model.zero_grad()
        loss = criterion(model(x), labels)
        loss.backward()
        x = x + ALPHA * x.grad.sign()
        x = torch.max(torch.min(x, images + EPS), images - EPS).clamp(0, 1)
    return x.detach()

In [30]:
testloader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('./data', train=False, download=False, transform=transforms.ToTensor()), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [31]:
import gc

def generate_and_save(name, testloader):
    print(f'--- Generating for {name} ---')
    torch.cuda.empty_cache()
    model = get_model(name)
    model.load_state_dict(torch.load(MODEL_PATHS[name], map_location='cpu'))
    model = model.to(DEVICE)
    model.eval()
    criterion = nn.CrossEntropyLoss()

    fgsm_list, pgd_list, labels_list = [], [], []

    for images, labels in testloader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        fgsm_batch = fgsm(model, images, labels, criterion).cpu()
        pgd_batch  = pgd(model, images, labels, criterion).cpu()
        fgsm_list.append(fgsm_batch)
        pgd_list.append(pgd_batch)
        labels_list.append(labels.cpu())

        del images, labels, fgsm_batch, pgd_batch
        torch.cuda.empty_cache()

    os.makedirs('../adv_examples', exist_ok=True)
    torch.save({
        'fgsm': torch.cat(fgsm_list),
        'pgd': torch.cat(pgd_list),
        'labels': torch.cat(labels_list)
    }, ADV_PATHS[name])
    print(f'  Saved to {ADV_PATHS[name]}')

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [32]:
generate_and_save("vgg16", testloader )

--- Generating for vgg16 ---


AcceleratorError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
generate_and_save("densenet", testloader)

--- Generating for densenet ---


AcceleratorError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
generate_and_save("resnet34", testloader)

In [ ]:
generate_and_save("vgg11", testloader)

In [ ]:
generate_and_save("resnet18", testloader)